# Explore SAS XPORT (`.xpt`) data

Generic notebook for any NHANES / SAS XPORT file.

1. Drop `.xpt` files into the project `data/` folder.
2. Set **`XPT_NAME`** (filename) or **`XPT_PATH`** (full path) in the config cell.
3. Run all cells — tabular extract, profile, and exploratory plots.

Select kernel **HackNation6 (.venv)**.

## 1. Config — change the file here

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------------------------
# Choose which file to analyse
# ---------------------------------------------------------------------------
# Option A — filename inside the project's data/ folder:
XPT_NAME = "P_RHQ.xpt"

# Option B — absolute / relative path (set this to override XPT_NAME):
XPT_PATH = None  # e.g. Path.home() / "Downloads" / "DEMO_J.xpt"
# ---------------------------------------------------------------------------

# Resolve project root whether CWD is repo root or notebooks/
_candidates = [Path.cwd(), Path.cwd().parent, Path("..").resolve(), Path(".").resolve()]
ROOT = next(
    (p.resolve() for p in _candidates if (p / "data").is_dir() and (p / "requirements.txt").exists()),
    Path("..").resolve(),
)
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
%matplotlib inline
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

print(f"Project root : {ROOT}")
print(f"Data folder  : {DATA_DIR}")
print("Available .xpt files:")
xpt_files = sorted(DATA_DIR.glob("*.xpt"))
if xpt_files:
    for f in xpt_files:
        print(f"  • {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
else:
    print("  (none yet — copy files into data/)")


Project root : /Users/rajvasani/Developer/HackNation6
Data folder  : /Users/rajvasani/Developer/HackNation6/data
Available .xpt files:
  • P_RHQ.xpt  (1334 KB)


## 2. Load `.xpt` → DataFrame

In [ ]:
def load_xpt(path: Path) -> pd.DataFrame:
    """Read a SAS XPORT (.xpt) file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}\n"
            f"Put the .xpt in {DATA_DIR} or set XPT_PATH to a valid location."
        )
    try:
        import pyreadstat
        df, meta = pyreadstat.read_xport(str(path))
        # Attach column labels from the XPT metadata when available
        labels = getattr(meta, "column_names_to_labels", None) or {}
        df.attrs["column_labels"] = {k: v for k, v in labels.items() if v}
    except Exception:
        df = pd.read_sas(path, format="xport")
        df.attrs["column_labels"] = {}
    return df


path = Path(XPT_PATH) if XPT_PATH else (DATA_DIR / XPT_NAME)
df = load_xpt(path)

STEM = path.stem
print(f"Loaded : {path}")
print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## 3. Tabular overview

In [ ]:
labels = df.attrs.get("column_labels", {})

profile = pd.DataFrame(
    {
        "column": df.columns,
        "label": [labels.get(c, "") for c in df.columns],
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (100 * df.isna().mean().values).round(2),
        "nunique": [df[c].nunique(dropna=True) for c in df.columns],
    }
)
profile_path = OUT_DIR / f"{STEM}_profile.csv"
profile.to_csv(profile_path, index=False)
print(f"Wrote profile → {profile_path}")
profile


In [ ]:
df.describe(include="all").T

In [ ]:
# Full tabular extract
csv_path = OUT_DIR / f"{STEM}.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote CSV → {csv_path}  ({csv_path.stat().st_size / 1024:.0f} KB)")
df


## 4. Exploratory plots

Generic charts that adapt to whatever columns are in the file. Tweak `MAX_NUMERIC_PLOTS` / `MAX_CATEG_PLOTS` if needed.

In [ ]:
MAX_NUMERIC_PLOTS = 6   # histograms for numeric columns with most non-null values
MAX_CATEG_PLOTS = 6     # bar charts for low-cardinality columns
MAX_CATEG_LEVELS = 20   # skip columns with more distinct values than this


In [ ]:
# Missingness across all columns
miss = (df.isna().mean() * 100).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, max(4, 0.28 * len(miss))))
ax.barh(miss.index.astype(str), miss.values, color="#2a6f7a")
ax.set_xlabel("Missing (%)")
ax.set_title(f"Missingness — {STEM}")
fig.tight_layout()
plt.show()


In [ ]:
# Numeric distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Prefer columns that are actually populated
numeric_cols = sorted(numeric_cols, key=lambda c: df[c].notna().sum(), reverse=True)
numeric_cols = numeric_cols[:MAX_NUMERIC_PLOTS]

if not numeric_cols:
    print("No numeric columns to plot.")
else:
    n = len(numeric_cols)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, numeric_cols):
        s = df[col].dropna()
        ax.hist(s, bins=min(40, max(10, int(np.sqrt(len(s))))), color="#2a6f7a", edgecolor="white")
        title = labels.get(col, col)
        ax.set_title(title[:40] if title else col, fontsize=9)
        ax.set_xlabel(col, fontsize=8)
    for ax in axes[len(numeric_cols):]:
        ax.set_visible(False)
    fig.suptitle(f"Numeric distributions — {STEM}", y=1.01)
    fig.tight_layout()
    plt.show()


In [ ]:
# Low-cardinality / categorical-like columns
categ_cols = []
for col in df.columns:
    nunq = df[col].nunique(dropna=True)
    if 1 < nunq <= MAX_CATEG_LEVELS:
        categ_cols.append((col, nunq, df[col].notna().sum()))
categ_cols = sorted(categ_cols, key=lambda t: t[2], reverse=True)[:MAX_CATEG_PLOTS]

if not categ_cols:
    print("No low-cardinality columns to plot.")
else:
    n = len(categ_cols)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (col, _nunq, _) in zip(axes, categ_cols):
        counts = df[col].value_counts(dropna=False).head(MAX_CATEG_LEVELS)
        counts.plot(kind="bar", ax=ax, color="#c45c26", rot=45)
        title = labels.get(col, col)
        ax.set_title(title[:40] if title else col, fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("Count", fontsize=8)
    for ax in axes[len(categ_cols):]:
        ax.set_visible(False)
    fig.suptitle(f"Value counts — {STEM}", y=1.01)
    fig.tight_layout()
    plt.show()


## 5. Switch file

To analyse another dataset, change `XPT_NAME` / `XPT_PATH` in the config cell and **re-run all cells**.

Examples:

```python
XPT_NAME = "P_DEMO.xpt"          # another file in data/
XPT_PATH = Path("data/P_BMX.xpt")
XPT_PATH = Path.home() / "Downloads" / "RHQ_J.xpt"
```